# Zero-Shot Irony Classification with ModernBERT-base

**Approach**: ModernBERT-base is an encoder-only (BERT-style) model.  
It has no generative head, so "zero-shot" here means the **fill-mask (MLM) method**:

1. Append `"\nAnswer: [MASK]"` to every `full_prompt`
2. Run the MLM head → get logits over the full vocabulary at `[MASK]`
3. Compare the logit for token `"yes"` vs token `"no"`
4. `yes > no` → predicted **ironic (1)**, otherwise **non-ironic (0)**

No gradient updates. No labels seen at inference time.

## Step 1 — Create & Activate the Virtual Environment

Run these commands **in your terminal** (not inside the notebook).

```bash
# 1. Create a fresh venv (Python 3.10+ recommended)
python3 -m venv venv_zeroshot

# 2. Activate it
# macOS / Linux:
source venv_zeroshot/bin/activate
# Windows (PowerShell):
# .\venv_zeroshot\Scripts\Activate.ps1

# 3. Install dependencies
pip install --upgrade pip
pip install torch transformers>=4.48.0 pandas scikit-learn numpy jupyter

# 4. Launch Jupyter (from the same terminal, still inside the venv)
jupyter notebook
```

> **Why a fresh venv?**  
> ModernBERT requires `transformers >= 4.48.0`. A separate environment avoids  
> conflicts with the fine-tuning environment you used before.

## Step 2 — Verify the Environment

In [1]:
import sys, torch, transformers
print(f"Python     : {sys.version.split()[0]}")
print(f"PyTorch    : {torch.__version__}")
print(f"Transformers: {transformers.__version__}")

# transformers must be >= 4.48.0 for ModernBERT support
from packaging.version import Version
assert Version(transformers.__version__) >= Version("4.48.0"), (
    "Please upgrade: pip install transformers>=4.48.0"
)
print("All version checks passed ✓")

Python     : 3.14.2
PyTorch    : 2.10.0
Transformers: 5.3.0
All version checks passed ✓


## Step 3 — Imports & Configuration

In [2]:
import random
import numpy as np
import pandas as pd
import torch
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForMaskedLM
from sklearn.metrics import f1_score, accuracy_score, classification_report

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Configuration ─────────────────────────────────────────────────────────
CFG = {
    "model_name" : "answerdotai/ModernBERT-base",
    "input_col"  : "full_prompt",          # switch to "base_utterance" for ablation
    "max_length" : 256,                    # longer than training because prompts can be verbose
    "batch_size" : 16,                     # no gradients → larger batches are fine
    "device"     : "cuda" if torch.cuda.is_available() else "cpu",
    # Tokens to compare at [MASK] position
    "pos_token"  : "yes",                  # → ironic   (label = 1)
    "neg_token"  : "no",                   # → non-ironic (label = 0)
    # Suffix appended to every prompt so ModernBERT fills "yes" or "no"
    "mask_suffix": "\nAnswer: [MASK]",
}

RESULTS_FILE = "results_zeroshot.txt"

def log(text=""):
    print(text)
    with open(RESULTS_FILE, "a", encoding="utf-8") as f:
        f.write(text + "\n")

# Clear results file
with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    f.write("Zero-Shot Irony Classification Results\n")
    f.write(f"Run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("=" * 60 + "\n\n")

print(f"Device : {CFG['device']}")
print(f"Results: {RESULTS_FILE}")

Device : cpu
Results: results_zeroshot.txt


## Step 4 — Load Model & Tokenizer

`AutoModelForMaskedLM` loads ModernBERT with its **MLM head** (the head that
predicts masked tokens). This is the head we exploit for zero-shot scoring.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
model     = AutoModelForMaskedLM.from_pretrained(CFG["model_name"])
model     = model.to(CFG["device"])
model.eval()   # inference only — no dropout

# Pre-compute vocabulary IDs for "yes" and "no"
# tokenizer.encode returns [bos_id, token_id, eos_id] for most tokenizers;
# we keep only the middle token.
yes_id = tokenizer.encode(CFG["pos_token"], add_special_tokens=False)
no_id  = tokenizer.encode(CFG["neg_token"], add_special_tokens=False)

assert len(yes_id) == 1, f"'yes' tokenises to multiple tokens: {yes_id}"
assert len(no_id)  == 1, f"'no' tokenises to multiple tokens:  {no_id}"

YES_ID = yes_id[0]
NO_ID  = no_id[0]

print(f"'yes' token id : {YES_ID}")
print(f"'no'  token id : {NO_ID}")
print("Model loaded ✓")

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

'yes' token id : 9820
'no'  token id : 2369
Model loaded ✓


## Step 5 — Load Data

Same loading logic as the fine-tuning notebook, normalised to a common schema.

In [4]:
c1 = pd.read_csv("Condition1_context_richness_.csv", sep=";",
                  usecols=lambda c: not c.startswith("Unnamed"))
c2 = pd.read_csv("Condition2_common_ground_.csv",    sep=";")
c3 = pd.read_csv("Condition3_prompting_style_.csv",  sep=";")

c1 = c1.rename(columns={"context-level": "level", "basic_utterance": "base_utterance"})
c2 = c2.rename(columns={"cg_level": "level"})
c3 = c3.rename(columns={"prompt_level": "level"})

c1["condition"] = "C1_context_richness"
c2["condition"] = "C2_common_ground"
c3["condition"] = "C3_prompting_style"

COLS = ["item_id", "condition", "level", "irony_label", "base_utterance", "full_prompt"]
df = pd.concat([c1[COLS], c2[COLS], c3[COLS]], ignore_index=True)

df = df.dropna(subset=["irony_label", CFG["input_col"]]).reset_index(drop=True)
df["label"] = (df["irony_label"] == "ironic").astype(int)  # ironic=1, non-ironic=0

print(f"Total rows: {len(df)}")
print(df[["condition", "level", "irony_label"]].value_counts().sort_index().to_string())

Total rows: 220
condition            level  irony_label
C1_context_richness  high   ironic         30
                            non-ironic     30
                     low    ironic         30
                            non-ironic     30
C2_common_ground     high   ironic         15
                            non-ironic     15
                     low    ironic         15
                            non-ironic     15
C3_prompting_style   high   ironic         10
                            non-ironic     10
                     low    ironic         10
                            non-ironic     10


## Step 6 — Zero-Shot Scoring Function

For each item:
1. Append `"\nAnswer: [MASK]"` to the `full_prompt`
2. Tokenise and find the position of `[MASK]`
3. Run the MLM head → logits shape `(seq_len, vocab_size)` at the `[MASK]` position
4. Compare `logit[YES_ID]` vs `logit[NO_ID]`  
   → `yes > no` means the model judges it **ironic**

In [5]:
def zero_shot_predict(texts, tokenizer, model, yes_id, no_id,
                      mask_suffix, max_length, batch_size, device):
    """
    Parameters
    ----------
    texts : list[str]   — raw input strings (full_prompt or base_utterance)

    Returns
    -------
    preds  : list[int]  — 1 = ironic, 0 = non-ironic
    scores : list[float] — confidence = logit_yes - logit_no (higher → more ironic)
    """
    mask_token = tokenizer.mask_token   # "[MASK]" for BERT-style tokenizers
    prompted   = [t + mask_suffix.replace("[MASK]", mask_token) for t in texts]

    preds, scores = [], []

    for start in range(0, len(prompted), batch_size):
        batch_texts = prompted[start : start + batch_size]

        enc = tokenizer(
            batch_texts,
            max_length=max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        input_ids      = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)

        with torch.no_grad():
            logits = model(input_ids=input_ids,
                           attention_mask=attention_mask).logits
            # logits shape: (batch, seq_len, vocab_size)

        for i in range(input_ids.size(0)):
            ids = input_ids[i].tolist()
            # Find the [MASK] position — there should be exactly one
            mask_positions = [j for j, tok in enumerate(ids)
                              if tok == tokenizer.mask_token_id]
            if not mask_positions:
                # Prompt was truncated and [MASK] fell off — append prediction 0
                preds.append(0)
                scores.append(0.0)
                continue

            mask_pos      = mask_positions[0]
            token_logits  = logits[i, mask_pos]          # (vocab_size,)
            yes_logit     = token_logits[yes_id].item()
            no_logit      = token_logits[no_id].item()

            preds.append(1 if yes_logit > no_logit else 0)
            scores.append(yes_logit - no_logit)          # positive → model leans ironic

    return preds, scores

print("Scoring function defined ✓")

Scoring function defined ✓


## Step 7 — Quick Sanity Check (2 examples)

In [6]:
test_examples = [
    # Clearly ironic
    'Is the following statement ironic? Answer with yes or no.\n "What a great day to spill coffee all over myself."',
    # Clearly literal
    'Is the following statement ironic? Answer with yes or no.\n "The sun is shining today."',
]

p, s = zero_shot_predict(
    test_examples, tokenizer, model,
    YES_ID, NO_ID,
    CFG["mask_suffix"], CFG["max_length"],
    batch_size=2, device=CFG["device"]
)

for text, pred, score in zip(test_examples, p, s):
    label = "ironic" if pred == 1 else "non-ironic"
    print(f"Pred: {label:12s}  score(yes-no): {score:+.3f}")
    print(f"  {text[:80]}...")
    print()

Pred: non-ironic    score(yes-no): -0.820
  Is the following statement ironic? Answer with yes or no.
 "What a great day to ...

Pred: non-ironic    score(yes-no): -0.407
  Is the following statement ironic? Answer with yes or no.
 "The sun is shining t...



## Step 8 — Run Zero-Shot Inference on All Conditions

We evaluate the **whole dataset** — there is no train/val/test split needed  
for zero-shot because the model never sees labels.

In [7]:
# ── Run inference condition by condition ──────────────────────────────────
condition_results = {}

log("[ZERO-SHOT RESULTS — CONDITION SUMMARY]")
log(f"{'Condition':<25}  {'N':>4}  {'Macro F1':>9}  {'Accuracy':>9}  "
    f"{'F1 ironic':>10}  {'F1 non-ironic':>14}")
log("-" * 82)

for cond_name, group in df.groupby("condition"):
    texts  = group[CFG["input_col"]].tolist()
    labels = group["label"].tolist()

    preds, scores = zero_shot_predict(
        texts, tokenizer, model,
        YES_ID, NO_ID,
        CFG["mask_suffix"], CFG["max_length"],
        CFG["batch_size"], CFG["device"]
    )

    group = group.copy()
    group["pred"]  = preds
    group["score"] = scores
    condition_results[cond_name] = group

    macro_f1     = f1_score(labels, preds, average="macro")
    acc          = accuracy_score(labels, preds)
    f1_ironic    = f1_score(labels, preds, pos_label=1, average="binary")
    f1_nonironic = f1_score(labels, preds, pos_label=0, average="binary")

    log(f"{cond_name:<25}  {len(group):>4}  {macro_f1:>9.4f}  {acc:>9.4f}  "
        f"{f1_ironic:>10.4f}  {f1_nonironic:>14.4f}")

log()

[ZERO-SHOT RESULTS — CONDITION SUMMARY]
Condition                     N   Macro F1   Accuracy   F1 ironic   F1 non-ironic
----------------------------------------------------------------------------------
C1_context_richness         120     0.3076     0.3250      0.1980          0.4173
C2_common_ground             60     0.4151     0.5167      0.1714          0.6588
C3_prompting_style           40     0.4505     0.5000      0.2857          0.6154



## Step 9 — Per-Level Breakdown

In [8]:
log("[ZERO-SHOT RESULTS — PER LEVEL BREAKDOWN]")
log(f"{'Condition':<25}  {'Level':<8}  {'N':>4}  {'Macro F1':>9}  {'Accuracy':>9}")
log("-" * 65)

for cond_name, group in condition_results.items():
    for level, grp in group.groupby("level"):
        lf1  = f1_score(grp["label"], grp["pred"], average="macro")
        lacc = accuracy_score(grp["label"], grp["pred"])
        log(f"{cond_name:<25}  {level:<8}  {len(grp):>4}  {lf1:>9.4f}  {lacc:>9.4f}")

log()

[ZERO-SHOT RESULTS — PER LEVEL BREAKDOWN]
Condition                  Level        N   Macro F1   Accuracy
-----------------------------------------------------------------
C1_context_richness        high        60     0.1498     0.1500
C1_context_richness        low         60     0.4375     0.5000
C2_common_ground           high        30     0.4034     0.5333
C2_common_ground           low         30     0.4223     0.5000
C3_prompting_style         high        20     0.4792     0.5000
C3_prompting_style         low         20     0.4048     0.5000



## Step 10 — Detailed Classification Report per Condition

In [9]:
for cond_name, group in condition_results.items():
    labels = group["label"].tolist()
    preds  = group["pred"].tolist()

    print(f"\n{'='*60}")
    print(f"  {cond_name}  (n={len(group)})")
    print(classification_report(labels, preds,
                                 target_names=["non-ironic", "ironic"]))


  C1_context_richness  (n=120)
              precision    recall  f1-score   support

  non-ironic       0.37      0.48      0.42        60
      ironic       0.24      0.17      0.20        60

    accuracy                           0.33       120
   macro avg       0.31      0.33      0.31       120
weighted avg       0.31      0.33      0.31       120


  C2_common_ground  (n=60)
              precision    recall  f1-score   support

  non-ironic       0.51      0.93      0.66        30
      ironic       0.60      0.10      0.17        30

    accuracy                           0.52        60
   macro avg       0.55      0.52      0.42        60
weighted avg       0.55      0.52      0.42        60


  C3_prompting_style  (n=40)
              precision    recall  f1-score   support

  non-ironic       0.50      0.80      0.62        20
      ironic       0.50      0.20      0.29        20

    accuracy                           0.50        40
   macro avg       0.50      0.50     

## Step 11 — Save Per-Item Predictions to results_zeroshot.txt

In [10]:
label_map = {1: "ironic", 0: "non-ironic"}

log("[ZERO-SHOT RESULTS — PER ITEM PREDICTIONS]")

for cond_name, group in condition_results.items():
    log(f"\n  Condition: {cond_name}")
    log(f"  {'item_id':<15}  {'level':<8}  {'true label':<12}  "
        f"{'predicted':<12}  {'score':>8}  {'correct'}")
    log("  " + "-" * 72)

    for _, row in group.iterrows():
        correct = "YES" if row["label"] == row["pred"] else "NO"
        log(f"  {row['item_id']:<15}  {row['level']:<8}  "
            f"{label_map[row['label']]:<12}  "
            f"{label_map[row['pred']]:<12}  "
            f"{row['score']:>+8.3f}  {correct}")

log()
log(f"Run finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nAll results saved to: {RESULTS_FILE}")

[ZERO-SHOT RESULTS — PER ITEM PREDICTIONS]

  Condition: C1_context_richness
  item_id          level     true label    predicted        score  correct
  ------------------------------------------------------------------------
  C1_01_L          low       ironic        non-ironic      -0.200  NO
  C1_01_L_NI       low       non-ironic    non-ironic      -0.200  YES
  C1_01_H          high      ironic        non-ironic      -0.517  NO
  C1_01_H_NI       high      non-ironic    ironic          +0.415  NO
  C1_02_L          low       ironic        non-ironic      -0.348  NO
  C1_02_L_NI       low       non-ironic    non-ironic      -0.348  YES
  C1_02_H          high      ironic        non-ironic      -0.841  NO
  C1_02_H_NI       high      non-ironic    ironic          +0.066  NO
  C1_03_L          low       ironic        non-ironic      -0.034  NO
  C1_03_L_NI       low       non-ironic    non-ironic      -0.034  YES
  C1_03_H          high      ironic        non-ironic      -0.241  NO


## Step 12 — Optional Ablation: `base_utterance` as Input

Re-run with just the raw utterance (no prompt framing) to measure  
how much the prompt structure helps the MLM head.

In [11]:
# Uncomment and run to compare against full_prompt results
#
# CFG["input_col"] = "base_utterance"
#
# for cond_name, group in df.groupby("condition"):
#     texts  = group["base_utterance"].tolist()
#     labels = group["label"].tolist()
#     preds, scores = zero_shot_predict(
#         texts, tokenizer, model,
#         YES_ID, NO_ID,
#         CFG["mask_suffix"], CFG["max_length"],
#         CFG["batch_size"], CFG["device"]
#     )
#     macro_f1 = f1_score(labels, preds, average="macro")
#     acc      = accuracy_score(labels, preds)
#     print(f"{cond_name}: Macro F1={macro_f1:.4f}  Acc={acc:.4f}")

print("Ablation cell ready — uncomment to run.")

Ablation cell ready — uncomment to run.


## Done

`results_zeroshot.txt` contains three sections:

| Section | Contents |
|---|---|
| `[ZERO-SHOT RESULTS — CONDITION SUMMARY]` | Macro F1, Accuracy, per-class F1 for each condition |
| `[ZERO-SHOT RESULTS — PER LEVEL BREAKDOWN]` | Metrics split by `low` / `high` level |
| `[ZERO-SHOT RESULTS — PER ITEM PREDICTIONS]` | Every item with true label, predicted label, and score |

---

### How the score column works
`score = logit(yes) − logit(no)`  
- Large positive value → model is confident the utterance **is ironic**  
- Large negative value → model is confident it is **not ironic**  
- Values near 0 → uncertain

---

### Limitations of the fill-mask zero-shot approach
- ModernBERT-base was pre-trained on text prediction, **not** irony detection.  
  Expect performance close to chance (macro F1 ≈ 0.50) as a baseline.
- The method is sensitive to the suffix wording. If you want to experiment,  
  try `"The answer is [MASK]"` or `"[MASK], this is ironic."` as alternatives.
- Comparing against your fine-tuned model shows the value of supervision.